In [ ]:
from __future__ import annotations

import os
import shutil
import warnings

import numpy as np
import lightning as pl
from functools import partial
from dgl.data.utils import split_dataset
#from mp_api.client import MPRester
from pytorch_lightning.loggers import CSVLogger

import matgl
from matgl.ext.pymatgen import Structure2Graph, get_element_list
from matgl.graph.data import MGLDataset, MGLDataLoader, collate_fn_pes
from matgl.models import M3GNet
from matgl.utils.training import PotentialLightningModule
from matgl.config import DEFAULT_ELEMENTS

# To suppress warnings for clearer output
warnings.simplefilter("ignore")

In [ ]:
# from ase.io.trajectory import Trajectory

# training_trajs = []
# for i in range(1, 11):
#     training_trajs += list(Trajectory(f"./Data/Training/nonLiPS_box{i}.traj"))
#     training_trajs += list(Trajectory(f"./Data/Training/LiPS_box{i}.traj"))

# testing_trajs = []
# for i in range(1,2):
#     testing_trajs += list(Trajectory(f"./Data/Testing/LiPS_box{i}.traj"))
#     testing_trajs += list(Trajectory(f"./Data/Testing/nonLiPS_box{i}.traj"))

# print("Training Data Length: " + str(len(training_trajs)))
# print("Testing Data Length: " + str(len(testing_trajs)))

In [ ]:
# print("Training Data Length: " + str(len(training_trajs)))
# print("Testing Data Length: " + str(len(testing_trajs)))

In [ ]:
# from ase.io import write

# write("train.traj", training_trajs)

# # Assuming `testing_trajs` is already loaded as a list of frames
# selected_frames = []

# # Select the first every 2nd frame from the first 6000 frames
# selected_frames.extend(testing_trajs[:6000:10])

# # Select every other frame from the last 2000 frames
# selected_frames.extend(testing_trajs[-2000::10])

# # Write the selected frames to a .traj file
# write("test.traj", selected_frames)

In [ ]:
import random
import torch

# very important for multi-gpu training, do not delete
torch.manual_seed(0)

torch.cuda.empty_cache()

In [ ]:
from ase.io.trajectory import Trajectory
# Initialize atoms_list as an empty list
atoms_list = Trajectory('train.traj')

print("atoms_list length: " + str(len(atoms_list)))

In [ ]:
import numpy as np
from ase.units import GPa

def custom_voigt_to_tensor(voigt_array):
    """
    Convert a stress tensor from custom Voigt-like notation to a 3x3 matrix.
    The order is:
      1. Diagonal (σ11, σ22, σ33)
      2. Off-diagonal: top-right (σ12), top-left (σ13), and right-left (σ23).

    Parameters:
    voigt_array (array-like): 1D array with 6 components in the specified order.

    Returns:
    np.ndarray: 3x3 stress tensor matrix.
    """
    if len(voigt_array) != 6:
        raise ValueError("Input array must have 6 components.")

    tensor = np.zeros((3, 3))
    tensor[0, 0] = voigt_array[0]  # σ11
    tensor[1, 1] = voigt_array[1]  # σ22
    tensor[2, 2] = voigt_array[2]  # σ33
    tensor[0, 1] = tensor[1, 0] = voigt_array[5]  # σ12
    tensor[0, 2] = tensor[2, 0] = voigt_array[4]  # σ13
    tensor[1, 2] = tensor[2, 1] = voigt_array[3]  # σ23

    return tensor/(GPa)

In [ ]:
import concurrent.futures
import time
#from pymatgen.io import vasp
import numpy as np
import pymatgen.io.ase

start = time.process_time() 

# Example usage
voigt_array = np.array(atoms_list[0].get_stress())
tensor = custom_voigt_to_tensor(voigt_array)
print("Stress Tensor (3x3 Matrix):")
print(tensor)

def process_atom(atom):
    structure = pymatgen.io.ase.AseAtomsAdaptor.get_structure(atom)
    forces = atom.get_forces()[None, :, :]
    energy = atom.get_potential_energy()
    stresses = custom_voigt_to_tensor(atom.get_stress()).tolist()
    return structure, forces, energy, stresses

# Initialize lists to store results
structure_list = []
forces_list = []
energies_list = []
stresses_list = []

# Use ProcessPoolExecutor to parallelize the processing
with concurrent.futures.ProcessPoolExecutor(max_workers=16) as executor:
    results = list(executor.map(process_atom, atoms_list))

# Unpack the results
for structure, forces, energy, stresses in results:
    structure_list.append(structure)
    forces_list.append(forces)
    energies_list.append(energy)
    stresses_list.append(stresses)

print(time.process_time() - start)

In [ ]:
# Perform the final concatenation and conversion
baseline_forces = [force[0] for force in forces_list]
baseline_energies = energies_list
baseline_stresses = stresses_list

# Print lengths to verify
print("structure_list length")
print(len(structure_list))
print("baseline_forces length")
print(len(baseline_forces))
print("baseline_energies length")
print(len(baseline_energies))
print("baseline_stresses length")
print(len(baseline_stresses))

adjusted_energies = [
    energy / len(structure) 
    for energy, structure in zip(baseline_energies, structure_list)
]

labels = {
    "energies": adjusted_energies,
    "forces": baseline_forces,
    #"magmoms": magmoms,
    "stresses": baseline_stresses,
}

In [ ]:
atoms_list_val = Trajectory('test.traj')

In [ ]:
import concurrent.futures
import time
from pymatgen.io import vasp
import numpy as np
import pymatgen.io.ase

start = time.process_time() 

# Initialize lists to store results
structure_list_val = []
forces_list_val = []
energies_list_val = []
stresses_list_val = []

# Use ProcessPoolExecutor to parallelize the processing
with concurrent.futures.ProcessPoolExecutor(max_workers=16) as executor:
    results_val = list(executor.map(process_atom, atoms_list_val))

# Unpack the results
for structure_val, forces_val, energy_val, stresses_val in results_val:
    structure_list_val.append(structure_val)
    forces_list_val.append(forces_val)
    energies_list_val.append(energy_val)
    stresses_list_val.append(stresses_val)

print(time.process_time() - start)

In [ ]:
# Perform the final concatenation and conversion
baseline_forces_val = [force[0] for force in forces_list_val]
baseline_energies_val = energies_list_val
baseline_stresses_val = stresses_list_val

# Print lengths to verify
print("structure_list length")
print(len(structure_list_val))
print("baseline_forces length")
print(len(baseline_forces_val))
print("baseline_energies length")
print(len(baseline_energies_val))
print("baseline_stresses length")
print(len(baseline_stresses_val))

adjusted_energies_val = [
    energy / len(structure) 
    for energy, structure in zip(baseline_energies_val, structure_list_val)
]

labels_val = {
    "energies": adjusted_energies_val,
    "forces": baseline_forces_val,
    #"magmoms": magmoms,
    "stresses": baseline_stresses_val,
}

In [ ]:
baseline_stresses_val[0]

In [ ]:
element_types = DEFAULT_ELEMENTS
converter = Structure2Graph(element_types=element_types, cutoff=5.0)

def convert_to_serializable(obj):
    if isinstance(obj, dict):
        return {key: convert_to_serializable(value) for key, value in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(element) for element in obj]
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    else:
        return obj

labels = convert_to_serializable(labels)

dataset = MGLDataset(
    save_dir='./Data/save_train_best',
    threebody_cutoff=4.0, 
    structures=structure_list, 
    converter=converter, 
    labels=labels, 
    include_line_graph=True
)

labels_val = convert_to_serializable(labels_val)

dataset_val = MGLDataset(
    save_dir='./Data/save_val_best',
    raw_dir='./Data/raw',
    threebody_cutoff=4.0, 
    structures=structure_list_val, 
    converter=converter, 
    labels=labels_val, 
    include_line_graph=True
)

train_data, val_data1, test_data1 = split_dataset(
    dataset,
    frac_list=[1,0,0],
    shuffle=True,
    random_state=42,
)

train_data1, val_data, test_data1 = split_dataset(
    dataset_val,
    frac_list=[0,1,0],
    shuffle=False,
    random_state=42,
)


In [ ]:
# if you are not intended to use stress for training, switch include_stress=False!
my_collate_fn = partial(collate_fn_pes, include_line_graph=True, include_stress=True)
train_loader, val_loader = MGLDataLoader(
    train_data=train_data,
    val_data=val_data,
    collate_fn=my_collate_fn,
    batch_size=21,
    num_workers=8,
    pin_memory=True,
    generator=torch.Generator(device='cpu')
)

In [ ]:
import torch
from torch.optim.lr_scheduler import ReduceLROnPlateau
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping, Callback
from lightning.pytorch.loggers import CSVLogger
import lightning.pytorch as pl
import matgl

# download a pre-trained M3GNet
m3gnet_nnp = matgl.load_model("M3GNet-MP-2021.2.8-PES")
model_pretrained = m3gnet_nnp.model

property_offset = m3gnet_nnp.element_refs.property_offset

# Define optimizer using Adam with weight decay for regularization
optimizer = torch.optim.Adam(model_pretrained.parameters(), lr=1e-3, weight_decay=1e-5)

# Custom callback to adjust stress weight over epochs
class AdjustStressWeightCallback(Callback):
    def __init__(self, initial_weight, max_weight, growth_rate, decay_epoch_start):
        self.initial_weight = initial_weight
        self.max_weight = max_weight
        self.growth_rate = growth_rate
        self.decay_epoch_start = decay_epoch_start

    def on_epoch_end(self, trainer, pl_module):
        # Gradually increase the stress weight each epoch until decay starts
        if trainer.current_epoch < self.decay_epoch_start:
            new_weight = min(self.initial_weight * (1 + self.growth_rate) ** trainer.current_epoch, self.max_weight)
        else:
            # Gradually decrease the stress weight back to the initial value
            new_weight = max(self.max_weight * (1 - self.growth_rate) ** (trainer.current_epoch - self.decay_epoch_start), self.initial_weight)
        pl_module.stress_weight = new_weight
        print(f"Epoch {trainer.current_epoch}: Adjusted stress weight to {new_weight}")

# Lightning Module
class FineTunePotentialModule(PotentialLightningModule):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def configure_optimizers(self):
        scheduler = {
            'scheduler': ReduceLROnPlateau(
                self.optimizer, mode='min', factor=0.1, patience=3, verbose=True, min_lr=1e-6
            ),
            'monitor': 'val_Total_Loss'  # Change to monitor a valid metric
        }
        return [self.optimizer], [scheduler]

    def on_train_epoch_end(self):
        """Step scheduler with validation loss."""
        # Retrieve the validation loss from logged metrics
        val_loss = self.trainer.callback_metrics.get("val_Total_Loss")  # Use val_Total_Loss
        if val_loss is not None:
            sch = self.lr_schedulers()
            sch.step(val_loss)  # Pass validation loss to scheduler
        else:
            print("Warning: Validation loss not available to pass to the scheduler.")

# Instantiate the fine-tuning model
initial_stress_weight = 0.1  # Start with the original stress weight
lit_module_finetune = FineTunePotentialModule(
    model=model_pretrained,
    element_refs=property_offset,
    lr=1e-3,
    include_line_graph=True,
    stress_weight=initial_stress_weight,
    optimizer=optimizer,
)

# Early stopping callback
early_stopping_callback = EarlyStopping(
    monitor="val_Total_Loss",  # Change to a valid metric
    patience=5,
    min_delta=1e-4,
    mode='min',
    verbose=True,
)

# Model checkpoint callback
checkpoint_callback = ModelCheckpoint(
    dirpath="./model_checkpoints",
    filename="30-m3gnet_fa_classifier_{epoch:02d}-{val_Total_Loss:.6f}",  # Update to reflect monitored metric
    every_n_epochs=1,
    save_top_k=3,  # Save only the top 3 models to save disk space
    monitor='val_Total_Loss',  # Monitor the same metric for saving checkpoints
    mode='min',
)

# Set parameters for stress weight adjustment
max_stress_weight = 1.0      # Maximum stress weight to reach
growth_rate = 0.1            # Increase/decrease stress weight by 10% each epoch
decay_epoch_start = 15       # Start decreasing stress weight after epoch 15

# Set seed for reproducibility
torch.manual_seed(0)

# Empty GPU cache before starting training
torch.cuda.empty_cache()

# Set matrix multiplication precision for performance
torch.set_float32_matmul_precision('high')

# Logger setup
csv_logger = CSVLogger("logs", name="M3GNet_finetuning")

# Trainer setup with early stopping, checkpoint, gradient accumulation, and custom callback
trainer = pl.Trainer(
    accumulate_grad_batches=2,  # Accumulate gradients to simulate larger batch size (batch size of 32)
    precision=32,  # Use full precision training for compatibility
    callbacks=[
        checkpoint_callback,
        early_stopping_callback,
        AdjustStressWeightCallback(initial_stress_weight, max_stress_weight, growth_rate, decay_epoch_start),
    ],
    accelerator='gpu',
    devices=1,
    max_epochs=30,
    logger=[csv_logger],
    inference_mode=False,
)

trainer.fit(
    model=lit_module_finetune,
    ckpt_path=f"./model_checkpoints/30-m3gnet_fa_classifier_epoch=07-val_Total_Loss=2.022909.ckpt",
    train_dataloaders=train_loader,
    val_dataloaders=val_loader,
)

In [ ]:
# # download a pre-trained M3GNet
# m3gnet_nnp = matgl.load_model("M3GNet-MP-2021.2.8-PES")
# model_pretrained = m3gnet_nnp.model
# # you should test whether including the original property_offset helps improve training and validation accuracy
# lit_module_finetune = PotentialLightningModule(model=model_pretrained, lr=1e-4, include_line_graph=True, stress_weight = 0.1)

In [ ]:
# from lightning.pytorch.callbacks import ModelCheckpoint
# val_every_n_epochs = 1

# checkpoint_callback = ModelCheckpoint(
#     dirpath="./model_checkpoints", # <--- specify this on the trainer itself for version control
#     filename="50-m3gnet_fa_classifier_{epoch:02d}-{val_loss:.6f}",
#     every_n_epochs=val_every_n_epochs,
#     save_top_k=-1,  # <--- this is important!
# )

In [ ]:
# # very important for multi-gpu training, do not delete
# torch.manual_seed(0)

# torch.cuda.empty_cache()

# # If you wish to disable GPU or MPS (M1 mac) training, use the accelerator="cpu" kwarg.
# torch.set_float32_matmul_precision('medium')
# logger = CSVLogger("logs", name="M3GNet_finetuning")
# trainer = pl.Trainer(
#     accumulate_grad_batches=1,
#     callbacks=[checkpoint_callback], 
#     accelerator='gpu', 
#     devices=1, 
#     max_epochs=50, 
#     logger=logger, 
#     inference_mode=False, 
#     #strategy='ddp_notebook'
# )

In [ ]:
# torch.cuda.empty_cache()
# trainer.fit(
#     model=lit_module_finetune, 
#     #ckpt_path=f"./model_checkpoints_2/50-m3gnet_fa_classifier_epoch=48-val_loss=0.000000.ckpt",
#     train_dataloaders=train_loader, 
#     val_dataloaders=val_loader,
# )

In [ ]:
# save trained model
model_save_path = "./big_boy_m3gnet/"
lit_module_finetune.model.save(model_save_path)
# load trained model
trained_model = matgl.load_model(path=model_save_path)